<a href="https://colab.research.google.com/github/Ebrardemir/amazon-sentiment-analysis/blob/main/notebooks/06_aray%C3%BCz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duygu Analizi Modeli Canlı Test Arayüzü (Gradio)

Bu defterde, Amazon Dataset Reviews 2023 içerisinden aldığımız Electronics kategorisindeki veriler kullanılarak eğitilen en başarılı modelimizin, basit bir web arayüzü üzerinden canlı (real-time) olarak test edilmesi sağlanmıştır. Önceki defterlerde örneklem çıkarılıp, veri seti üstünde analiz yapılmış, ön işleme adımları gerçekleştirilmiş ve farklı makine öğrenmesi modelleri (TF-IDF, BERT vb.) eğitilerek en iyi performansı gösteren model dışa aktarılmıştır. Bu defterde ise eğitilen şampiyon model ayaklandırılarak yeni ve hiç görülmemiş veriler üzerinde tahmin yapması sağlanmaktadır.

### Yapılan Adımlar:

1. **Kayıtlı Modellerin Yüklenmesi:** Önceki eğitim aşamalarında `joblib` kullanılarak Google Drive'a kaydedilen en iyi duygu analizi modeli ve metinleri sayısallaştıran vektörleyiciler (`tfidf_title` ve `tfidf_text`) ortama yüklenmiştir.
2. **Gradio ile Arayüz Tasarımı:** Kullanıcıdan ürün başlığı (title) ve ürün yorumu (text) alabilmek için `gradio` kütüphanesi kullanılarak interaktif bir web arayüzü oluşturulmuştur.
3. **Canlı Tahmin (Inference):** Kullanıcının girdiği yeni İngilizce metinler anlık olarak özellik matrisine dönüştürülüp modele sokulmuş ve "0 (Negatif)" veya "1 (Pozitif)" olarak duygu tahmini alınmıştır.
4. **Hata Yakalama ve Loglama (Flagging):** Modelin başarısız olduğu veya kararsız kaldığı uç durumların (edge cases) tespiti için bir "Flag" (İşaretleme) mekanizması kurulmuş ve bu spesifik girişlerin ileride incelenmek üzere Drive üzerinde bir CSV dosyasına (log) kaydedilmesi sağlanmıştır.

In [ ]:
!pip install gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gradio as gr
import joblib
from scipy.sparse import hstack

# 1. Google Drive'daki model yolları (Kendi dosya ismine göre 'best_04A_...' kısmını güncelle)
model_path = '/content/drive/MyDrive/Metin_Madenciliği/Models/best_04A_model_Logistic_Regression.joblib'
tfidf_title_path = '/content/drive/MyDrive/Metin_Madenciliği/Models/tfidf_title_04B.joblib'
tfidf_text_path = '/content/drive/MyDrive/Metin_Madenciliği/Models/tfidf_text_04B.joblib'

# 2. Modelleri yükleme
model = joblib.load(model_path)
tfidf_title = joblib.load(tfidf_title_path)
tfidf_text = joblib.load(tfidf_text_path)

# 3. Tahmin Fonksiyonu
def duygu_tahmin_et(baslik, yorum):
    # Boş giriş kontrolü
    if not baslik.strip() and not yorum.strip():
        return "Lütfen analiz için bir başlık veya yorum girin."

    # Metinleri TF-IDF ile vektörize etme
    title_vec = tfidf_title.transform([baslik])
    text_vec = tfidf_text.transform([yorum])

    # Matrisleri birleştirme
    combined_vec = hstack([title_vec, text_vec]).tocsr()

    # Modelden tahmini alma
    prediction = model.predict(combined_vec)[0]

    if prediction == 1:
        return " Pozitif (1) - Olumlu Yorum"
    else:
        return " Negatif (0) - Olumsuz Yorum"

# 4. Gradio Arayüzünü Oluşturma
arayuz = gr.Interface(
    fn=duygu_tahmin_et, # Çalışacak fonksiyon
    inputs=[
        gr.Textbox(label="Ürün Başlığı", placeholder="e.g., Excellent product!"),
        gr.Textbox(label="Ürün Yorumu", placeholder="e.g., The sound quality is amazing for the price.", lines=4)
    ],
    outputs=gr.Text(label="Tahmin Sonucu"),
    title="Amazon Elektronik: Duygu Analizi",
    description="Eğitilmiş modelinizi test etmek için bir başlık ve yorum girin.",
    flagging_dir="/content/drive/MyDrive/Metin_Madenciliği/flagged_logs"
)

# Arayüzü Colab hücresinin içinde başlat
arayuz.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://846b3ca9ce0e7e495d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Using existing dataset file at: /content/drive/MyDrive/Metin_Madenciliği/flagged_logs/dataset1.csv
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://846b3ca9ce0e7e495d.gradio.live
